In [ ]:
#Cell:1#
import os
import sys

# Use a relative path so it works on any computer
capstone_path = "./"
os.makedirs(capstone_path, exist_ok=True)

# Add the project folder to Python's path to allow importing helper scripts
sys.path.append(capstone_path)

print(f"Project path set to: {os.path.abspath(capstone_path)}")

In [ ]:
#Cell:3#
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# Import our custom function from the file created in Cell 2
try:
    from data_loader import extract_frames
    print("Successfully imported 'extract_frames' from data_loader.py.")
except ImportError:
    print("ERROR: Could not import 'extract_frames'. Check Cell 2 and sys.path.")

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#Cell:4#
class VideoFrameDataset(Dataset):
    """PyTorch Dataset for loading video frames."""
    def __init__(self, video_paths, labels, num_frames, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.num_frames = num_frames
        self.transform = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        # Define the standard size the 3D CNN expects
        RESIZE_TO = (224, 224)
        frames = extract_frames(video_path, max_frames=self.num_frames, resize=RESIZE_TO)

        if not frames: # If frames is an empty list
          print(f"Warning: Skipping {video_path} due to extraction error.")
          return None, None # Return None to be filtered later

        if self.transform:
            frames = self.transform(frames)

        return frames, label

# --- 1. Scan for video files ---
video_paths, labels = [], []
max_videos_per_class = 200 # Using a subset for faster testing

real_videos_path = os.path.join(capstone_path, "videos/Real_videos")
fake_videos_path = os.path.join(capstone_path, "videos/Deepfakes_videos")
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

print(f"Looking for real videos in: {real_videos_path}")
print(f"Looking for fake videos in: {fake_videos_path}")

# Process Real Videos
if os.path.exists(real_videos_path):
    count_real = 0
    for filename in os.listdir(real_videos_path):
        if filename.lower().endswith(video_extensions):
            video_paths.append(os.path.join(real_videos_path, filename))
            labels.append(0) # 0 for Real
            count_real += 1
            if count_real >= max_videos_per_class:
                break
    print(f"Found {count_real} real videos.")
else:
    print(f"WARNING: Real videos path does not exist: {real_videos_path}")

# Process Fake Videos
if os.path.exists(fake_videos_path):
    count_fake = 0
    for filename in os.listdir(fake_videos_path):
        if filename.lower().endswith(video_extensions):
            video_paths.append(os.path.join(fake_videos_path, filename))
            labels.append(1) # 1 for Fake
            count_fake += 1
            if count_fake >= max_videos_per_class:
                break
    print(f"Found {count_fake} fake videos.")
else:
    print(f"WARNING: Fake videos path does not exist: {fake_videos_path}")

# --- Define Transforms ---
NUM_FRAMES = 16
BATCH_SIZE = 4

# Define transforms to convert list of frames to a 3D-CNN-ready tensor
transform_3d = transforms.Compose([
    transforms.Lambda(lambda frames: torch.from_numpy(np.array(frames))),
    transforms.Lambda(lambda t: t.permute(3, 0, 1, 2)), # (N, H, W, C) -> (C, N, H, W)
    transforms.Lambda(lambda t: t.float() / 255.0),
])

# --- Create Dataset and DataLoaders ---
if len(video_paths) > 0:
    full_dataset = VideoFrameDataset(video_paths, labels, NUM_FRAMES, transform_3d)

    # Split 50% train, 25% val, 25% test
    total_size = len(full_dataset)
    train_size = int(total_size * 0.5)
    val_size = int(total_size * 0.25)
    test_size = total_size - train_size - val_size

    train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

    def collate_fn_skip_none(batch):
      """A custom collate_fn that filters out (None, None) samples."""
      batch = [item for item in batch if item[0] is not None]
      if not batch:
        # Return empty tensors if the entire batch was bad
        return torch.tensor([]), torch.tensor([])
      return torch.utils.data.dataloader.default_collate(batch)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn_skip_none)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_skip_none)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn_skip_none)

    print(f"Dataloaders created. [{len(train_dataset)} train | {len(val_dataset)} val | {len(test_dataset)} test samples]")
else:
    print("ERROR: No videos were found. The dataset is empty. Please check your file paths.")

In [ ]:
#Cell:5#
class Simple3DCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(Simple3DCNN, self).__init__()
        # Input shape: (C=3, N=16, H=224, W=224)

        self.conv1 = nn.Conv3d(3, 16, 3, padding=1)
        self.pool1 = nn.MaxPool3d((2, 2, 2)) # (16, 8, 112, 112)

        self.conv2 = nn.Conv3d(16, 32, 3, padding=1)
        self.pool2 = nn.MaxPool3d((2, 2, 2)) # (32, 4, 56, 56)

        self.conv3 = nn.Conv3d(32, 64, 3, padding=1)
        self.pool3 = nn.MaxPool3d((2, 2, 2)) # (64, 2, 28, 28)

        # Flattened size: 64 * 2 * 28 * 28
        self.fc1 = nn.Linear(64 * 2 * 28 * 28, 512) # Assuming 224x224 input frames
        self.fc2 = nn.Linear(512, num_classes)
        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = self.pool3(F.relu(self.conv3(x)))

        x = x.view(-1, 64 * 2 * 28 * 28) # Flatten

        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x) # No final softmax, CrossEntropyLoss will apply it
        return x

print("Model class 'Simple3DCNN' defined.")

In [ ]:
#Cell:6#
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Runs a single training epoch."""
    model.train()
    total_loss, total_correct, total_samples = 0, 0, 0

    for videos, labels in tqdm(dataloader, desc="Training"):
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * videos.size(0)
        _, preds = torch.max(outputs, 1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples * 100

def validate_one_epoch(model, dataloader, criterion, device):
    """Runs a single validation epoch."""
    model.eval()
    total_loss, total_correct, total_samples = 0, 0, 0

    with torch.no_grad():
        for videos, labels in tqdm(dataloader, desc="Validating"):
            videos, labels = videos.to(device), labels.to(device)

            outputs = model(videos)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * videos.size(0)
            _, preds = torch.max(outputs, 1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    return total_loss / total_samples, total_correct / total_samples * 100

# --- Setup and Main Loop ---
if 'train_loader' in locals():
    model_3d = Simple3DCNN(num_classes=2).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model_3d.parameters(), lr=0.001)
    NUM_EPOCHS = 30

    print(f"Starting 3D model training on {device}...")

    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_one_epoch(model_3d, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate_one_epoch(model_3d, val_loader, criterion, device)

        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% "
              f"| Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    print("Finished 3D Model Training.")

    # Optional: Save the baseline model
    # torch.save(model_3d.state_dict(), os.path.join(capstone_path, "my_3dcnn_model.pth"))
else:
    print("Skipping 3D model training as dataloaders were not created.")

In [ ]:
#Cell:7#
# ---Baseline Model Evaluation ---
# This cell calculates the detailed metrics for the baseline 3D CNN.

# --- Imports for evaluation ---
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

# --- function to also get probabilities ---
def get_predictions_and_probs_3d(model, data_loader, device):
    """
    Gets predictions, true labels, and probabilities for the 3D CNN model.
    """
    model = model.eval() # Set model to evaluation mode
    all_predictions = []
    all_true_labels = []
    all_probs = []

    with torch.no_grad():
        for videos, labels in tqdm(data_loader, desc="Testing 3D Model"):
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos) # This is our model_3d

            # --- Get probabilities ---
            probs = F.softmax(outputs, dim=1)

            # Get the class with the highest score (our prediction)
            _, preds = torch.max(outputs, 1)

            all_predictions.extend(preds.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

            # --- Store probability for the 'FAKE' class (index 1) ---
            all_probs.extend(probs[:, 1].cpu().numpy())

    # --- Return probabilities too ---
    return all_true_labels, all_predictions, all_probs

# --- 1. Get All Predictions ---
# Note: We use 'model_3d' and 'test_loader' from Cell 4 & 6
# ---  Get probabilities as well ---
y_true_3d, y_pred_3d, y_probs_3d = get_predictions_and_probs_3d(model_3d, test_loader, device)

# --- 2. Generate Classification Report ---
class_names = ["REAL", "FAKE"]
print("\n" + "="*30)
print("  BASELINE (3D CNN) CLASSIFICATION REPORT")
print("="*30)
print(classification_report(y_true_3d, y_pred_3d, target_names=class_names, zero_division=0))

# --- 3. Generate and Plot Confusion Matrix ---
print("\n" + "="*30)
print("    BASELINE (3D CNN) CONFUSION MATRIX")
print("="*30)
cm_3d = confusion_matrix(y_true_3d, y_pred_3d)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_3d, annot=True, fmt='d', cmap='Reds',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Baseline (3D CNN) Confusion Matrix')
plt.ylabel('Actual (True) Label')
plt.xlabel('Predicted Label')
plt.show()

# --- 4. Generate and Plot ROC Curve ---
print("\n" + "="*30)
print("     BASELINE (3D CNN) ROC CURVE & AUC")
print("="*30)

# Calculate ROC curve
fpr, tpr, _ = roc_curve(y_true_3d, y_probs_3d)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='red', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Baseline (3D CNN) ROC Curve')
plt.legend(loc="lower right")
plt.show()

print(f"Baseline AUC Score: {roc_auc:.4f}")

In [ ]:
#Cell:8#
def convert_frame_to_fft_spectrum(frame_rgb):
    """Converts a single RGB video frame to its log-magnitude spectrum."""
    # Convert the RGB frame to grayscale
    gray_frame = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY)

    # Apply 2D FFT and shift the zero-frequency component to the center
    f_transform = np.fft.fft2(gray_frame)
    f_transform_shifted = np.fft.fftshift(f_transform)

    # Calculate the log-magnitude spectrum for visualization
    magnitude_spectrum = np.log1p(np.abs(f_transform_shifted))

    return magnitude_spectrum

# --- Test and Verify the Function ---
sample_video_path = '/content/drive/MyDrive/Capstone/videos/Real_videos/BnzdgQ8SK9s.mp4'

if not os.path.exists(sample_video_path):
    print(f"Warning: Test video not found at {sample_video_path}. Using first video from dataset.")
    if 'video_paths' in locals() and len(video_paths) > 0:
        sample_video_path = video_paths[0]
    else:
        sample_video_path = None

if sample_video_path:
    # Use existing function to extract one frame
    original_frame = extract_frames(sample_video_path, max_frames=1)[0]

    # Use new function to get the frequency spectrum
    magnitude_spectrum = convert_frame_to_fft_spectrum(original_frame)

    # 3. Visualize the Output
    print("Displaying the original frame and its magnitude spectrum:")
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(original_frame)
    plt.title('Original RGB Frame')
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(magnitude_spectrum, cmap='gray')
    plt.title('Magnitude Spectrum')
    plt.axis('off')

    plt.show()
else:
    print("Skipping FFT visualization, no sample video found.")

In [ ]:
#Cell:9#
print("---Pipeline ---")

# 1. Define the new dataset paths
new_dataset_path = os.path.join(capstone_path, "frequency_dataset")
real_fft_path = os.path.join(new_dataset_path, "real")
fake_fft_path = os.path.join(new_dataset_path, "fake")

# 2. Create the new directories
os.makedirs(real_fft_path, exist_ok=True)
os.makedirs(fake_fft_path, exist_ok=True)
print(f"Created new dataset directories at: {new_dataset_path}")

# 3. Start the Processing Pipeline
# Use the collaboration plan: process a small batch first
# Set this to False to run the full dataset
IS_TEST_RUN = False
TEST_BATCH_SIZE = 20 # 20 real, 20 fake

if IS_TEST_RUN:
    print(f"--- RUNNING IN TEST MODE (First {TEST_BATCH_SIZE} real/fake) ---")
    real_indices = [i for i, label in enumerate(labels) if label == 0]
    fake_indices = [i for i, label in enumerate(labels) if label == 1]
    test_indices = real_indices[:TEST_BATCH_SIZE] + fake_indices[:TEST_BATCH_SIZE]
    processing_list = [(video_paths[i], labels[i]) for i in test_indices]
else:
    print(f"--- RUNNING IN FULL MODE ({len(video_paths)} videos) ---")
    processing_list = list(zip(video_paths, labels))

for video_path, label in tqdm(processing_list, desc="Processing videos"):
    try:
        video_filename = os.path.splitext(os.path.basename(video_path))[0]
        output_folder = real_fft_path if label == 0 else fake_fft_path

        # Extract frames from the video
        frames = extract_frames(video_path, max_frames=NUM_FRAMES)

        for i, frame_rgb in enumerate(frames):
            # 1. Use your existing function to get the spectrum
            magnitude_spectrum = convert_frame_to_fft_spectrum(frame_rgb)

            # 2. Normalize the array to a 0-255 integer range for saving as an image
            normalized_spectrum = cv2.normalize(magnitude_spectrum, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

            # 3. Define the new image filename
            new_image_filename = f"{video_filename}_frame_{i:02d}.png"
            new_image_path = os.path.join(output_folder, new_image_filename)

            # 4. Save the normalized array as a PNG image
            cv2.imwrite(new_image_path, normalized_spectrum)

    except Exception as e:
        print(f"Error processing {video_path}: {e}")

print("--- Pipeline Complete ---")
print(f"Frequency images saved in: {new_dataset_path}")

In [ ]:
#Cell:10#
# This cell redefines the dataset and transforms for Transfer Learning

import torchvision.models as models
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import os

# --- Define the new transform ---
# We use the default transforms for ResNet50, which include:
# - Resizing to 224x224
# - ToTensor()
# - Normalization with ImageNet mean and std dev
transfer_transform = models.ResNet50_Weights.DEFAULT.transforms()

print(f"Using new transforms for ResNet50:\n{transfer_transform}")

# --- Modify the FFTImageDataset ---
class FFTImageDataset(Dataset):
    """
    Modified dataset to load 2D FFT images for a pre-trained model.
    - Loads images
    - Converts grayscale to RGB (by stacking channels)
    - Applies specified transforms
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []

        # Scan 'real' and 'fake' subfolders
        for label, folder_name in enumerate(['real', 'fake']):
            folder_path = os.path.join(self.root_dir, folder_name)
            if not os.path.exists(folder_path):
                print(f"WARNING! Directory not found: {folder_path}")
                continue

            for filename in os.listdir(folder_path):
                if filename.endswith('.png'):
                    self.image_paths.append(os.path.join(folder_path, filename))
                    self.labels.append(label)

        print(f"Loaded {len(self.image_paths)} images from {self.root_dir}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        # Open as grayscale and convert to RGB (stacks channels)
        # This is the key change for the pre-trained model.
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

new_dataset_path = os.path.join(capstone_path, "frequency_dataset")

# --- Verification script (similar to Cell 9) ---
print("\n--- Starting verification script ---")

# 'new_dataset_path' was defined in Cell 8
# e.g., new_dataset_path = "/content/drive/MyDrive/Capstone/frequency_dataset"

try:
    fft_full_dataset = FFTImageDataset(root_dir=new_dataset_path,
                                       transform=transfer_transform)

    if len(fft_full_dataset) == 0:
        print("ERROR! FFTImageDataset is empty. Did 'Cell 8' run correctly?")
    else:
        # Create a test loader
        fft_loader = DataLoader(fft_full_dataset, batch_size=8, shuffle=True)
        images, labels = next(iter(fft_loader))

        print("\nVerification Successful!")
        print(f"Batch of images shape: {images.shape}")
        print(f"Batch of labels shape: {labels.shape}")
        print(f"Example shape (Batch, Channels, H, W): {images.shape}. (3 channels is correct)")

except Exception as e:
    print(f"\n--- Verification FAILED ---")
    print(f"Error: {e}")
    print("Please ensure 'new_dataset_path' is defined and Cell 8 ran.")

In [ ]:
#Cell:11#
# This cell defines the Transfer Learning model (ResNet50)

import torchvision.models as models
import torch.nn as nn

# 1. Load a pre-trained ResNet50 model
# We use the recommended 'weights' parameter
model_tl = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# 2. Freeze all the convolutional layers
# We don't want to retrain these, just use them as feature extractors
for param in model_tl.parameters():
    param.requires_grad = False

# 3. Replace the final classification layer (the 'fc' layer)
# Get the number of input features for the original 'fc' layer
num_ftrs = model_tl.fc.in_features

# Create a new, unfrozen final layer that outputs 2 classes
model_tl.fc = nn.Linear(num_ftrs, 2) # 2 classes: real, fake

# 4. Move the model to the device (defined in Cell 3)
model_tl = model_tl.to(device)

print("Successfully loaded pre-trained ResNet50.")
print("All layers frozen except the new final 'fc' layer.")
print(f"Model is on device: {device}")

In [ ]:
#Cell:12#
# This cell runs the training and validation for the Transfer Learning model

try:
    print("---Training Script (Transfer Learning) ---")

    # --- 1. Load the new 2D frequency dataset ---
    # We use the dataset class from our modified cell above
    # and the 'transfer_transform'
    fft_full_dataset = FFTImageDataset(root_dir=new_dataset_path,
                                       transform=transfer_transform)

    if len(fft_full_dataset) == 0:
        raise Exception("FFTImageDataset is empty. Did Cell 8 run correctly?")

    # --- 2. Split the dataset (50/25/25 Split) ---
    total_size = len(fft_full_dataset)
    train_size = int(total_size * 0.5)
    val_size = int(total_size * 0.25)
    test_size = total_size - train_size - val_size
    train_dataset_tl, val_dataset_tl, test_dataset_tl = random_split(fft_full_dataset,
                                                                   [train_size, val_size, test_size])
    print(f"Dataset split: {len(train_dataset_tl)} train, {len(val_dataset_tl)} val, {len(test_dataset_tl)} test")

    # --- 3. Create new DataLoaders ---
    BATCH_SIZE = 16 # This can be adjusted
    train_loader_tl = DataLoader(train_dataset_tl, batch_size=BATCH_SIZE, shuffle=True)
    val_loader_tl = DataLoader(val_dataset_tl, batch_size=BATCH_SIZE, shuffle=False)
    test_loader_tl = DataLoader(test_dataset_tl, batch_size=BATCH_SIZE, shuffle=False)
    print("2D DataLoaders created.")

    # --- 4. Setup and Main Loop ---
    # Use the ResNet50 model we defined in the cell above
    criterion_tl = nn.CrossEntropyLoss()

    # CRITICAL: Only pass the parameters of the new, unfrozen layer to the optimizer
    optimizer_tl = optim.Adam(model_tl.fc.parameters(), lr=0.001) # As suggested, lr=0.001

    NUM_EPOCHS = 30

    print(f"Starting 2D transfer learning for {NUM_EPOCHS} epochs on {device}...")

    # --- Run a full training ---
    # We reuse the train_one_epoch and validate_one_epoch functions from Cell 6
    for epoch in range(NUM_EPOCHS):

        # Train
        train_loss, train_acc = train_one_epoch(model_tl, train_loader_tl, criterion_tl, optimizer_tl, device)

        # Validate
        val_loss, val_acc = validate_one_epoch(model_tl, val_loader_tl, criterion_tl, device)

        print(f"Epoch [{epoch+1:02d}/{NUM_EPOCHS:02d}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%"
              f" | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

    print("\n--- Finished 2D Transfer Learning Model Training. ---")

    # Optional: Save the new model
    save_path = os.path.join(capstone_path, "my_resnet50_fft_model.pth")
    torch.save(model_tl.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

except Exception as e:
    print(f"\n--- ERROR: Could not run training script. ---")
    print(f"Error: {e}")




In [ ]:
#Cell:13#
# --- Intermediate Model Evaluation ---
# This cell evaluates the 'model_tl' we just trained in the previous cell.

# --- Imports for evaluation ---
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import numpy as np

# --- Define the prediction-getter function ---
def get_predictions_and_probs(model, data_loader, device):
    """
    Gets predictions, true labels, and probabilities from a model.
    """
    model = model.eval() # Set model to evaluation mode
    all_predictions = []
    all_true_labels = []
    all_probs = [] # List to store probabilities
    with torch.no_grad():
        for videos, labels in tqdm(data_loader, desc="Evaluating"):
            videos = videos.to(device)
            labels = labels.to(device)
            outputs = model(videos)
            probs = F.softmax(outputs, dim=1)
            _, preds = torch.max(outputs, 1)
            all_predictions.extend(preds.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    return all_true_labels, all_predictions, all_probs

print("\n--- Starting Intermediate Test Evaluation ---")

try:
    # --- 1. Get All Predictions & Probabilities ---
    # We use 'model_tl' and 'test_loader_tl' from the previous cell
    y_true_tl, y_pred_tl, y_probs_tl = get_predictions_and_probs(model_tl, test_loader_tl, device)

    # --- 2. Generate Classification Report (Precision, Recall, F1) ---
    class_names = ["REAL", "FAKE"]
    print("\n" + "="*30)
    print("  INTERMEDIATE (ResNet) CLASSIFICATION REPORT")
    print("="*30)
    print(classification_report(y_true_tl, y_pred_tl, target_names=class_names))

    # --- 3. Generate and Plot Confusion Matrix ---
    print("\n" + "="*30)
    print("    INTERMEDIATE (ResNet) CONFUSION MATRIX")
    print("="*30)
    cm_tl = confusion_matrix(y_true_tl, y_pred_tl)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Intermediate (ResNet) Confusion Matrix')
    plt.ylabel('Actual (True) Label')
    plt.xlabel('Predicted Label')
    plt.show()

    # --- 4. Generate and Plot ROC Curve ---
    print("\n" + "="*30)
    print("     INTERMEDIATE (ResNet) ROC CURVE & AUC")
    print("="*30)
    fpr, tpr, _ = roc_curve(y_true_tl, y_probs_tl)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Intermediate (ResNet) ROC Curve')
    plt.legend(loc="lower right")
    plt.show()
    print(f"Intermediate AUC Score: {roc_auc:.4f}")

except Exception as e:
    print(f"\n--- ERROR: Could not run evaluation script. ---")
    print(f"Error: {e}")

In [ ]:
#Cell:14#
# --- Setup for Fine-Tuning ---
# Assumes 'model_tl' is our 30-epoch trained model in memory

# 1. Unfreeze the last convolutional block of ResNet50 ('layer4')
print("--- Unfreezing 'layer4' for fine-tuning ---")

# Check the 'requires_grad' status before
print(f"Before: 'layer4' param requires_grad: {model_tl.layer4[0].conv1.weight.requires_grad}")

for param in model_tl.layer4.parameters():
    param.requires_grad = True

# Check the 'requires_grad' status after
print(f"After:  'layer4' param requires_grad: {model_tl.layer4[0].conv1.weight.requires_grad}")


# 2. Create a NEW optimizer with a VERY SMALL learning rate
# We use filter(lambda p: p.requires_grad, model_tl.parameters())
# to ensure we are only optimizing the unfrozen parameters
# (which are now layer4 and the final fc layer)

optimizer_ft = optim.Adam(
    filter(lambda p: p.requires_grad, model_tl.parameters()),
    lr=1e-5  # CRITICAL: Use a very small learning rate
)

print("\nNew optimizer created for fine-tuning with lr=1e-5")
print("Ready to start fine-tuning.")

In [ ]:
#Cell:15#
# --- Run Fine-Tuning Training ---

# We can continue for another 30 epochs
FT_EPOCHS = 30

# NOTE: We are continuing from the end of the last run.

print(f"--- Starting FINE-TUNING for {FT_EPOCHS} epochs on {device} ---")

# We re-use the same functions from Cell 6
# We re-use the same dataloaders (train_loader_tl, val_loader_tl)
# We re-use the same criterion (criterion_tl)
# We use the NEW optimizer (optimizer_ft)

for epoch in range(FT_EPOCHS):

    # Train
    train_loss, train_acc = train_one_epoch(model_tl, train_loader_tl, criterion_tl, optimizer_ft, device)

    # Validate
    val_loss, val_acc = validate_one_epoch(model_tl, val_loader_tl, criterion_tl, device)

    print(f"FT Epoch [{epoch+1:02d}/{FT_EPOCHS:02d}] | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}%"
          f" | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

print("\n--- Finished Fine-Tuning. ---")

# See if fine-tuning improved your final test score

print("\n--- Starting Final Test Evaluation (POST-FINE-TUNING) ---")
test_loss_ft, test_acc_ft = validate_one_epoch(model_tl, test_loader_tl, criterion_tl, device)

print(f"\nFINAL POST-FINE-TUNING TEST SET RESULTS:")
print(f"Test Loss: {test_loss_ft:.4f}")
print(f"Test Accuracy: {test_acc_ft:.2f}%")
print(f"(Previous test accuracy was 73.52%)")

In [ ]:
#Cell:16#
# --- Final Detailed Evaluation ---
# This cell calculates ALL detailed metrics for the final report.

import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm # Make sure tqdm is imported

def get_predictions_and_probs(model, data_loader, device):
    """
    Gets predictions, true labels, and probabilities from a model.
    """
    model = model.eval() # Set model to evaluation mode

    all_predictions = []
    all_true_labels = []
    all_probs = [] # List to store probabilities

    with torch.no_grad():
        for videos, labels in tqdm(data_loader, desc="Testing"):
            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            # Apply softmax to get probabilities
            probs = F.softmax(outputs, dim=1)

            # Get the class with the highest score (our prediction)
            _, preds = torch.max(outputs, 1)

            all_predictions.extend(preds.cpu().numpy())
            all_true_labels.extend(labels.cpu().numpy())

            # Get the probability of the 'FAKE' class (class 1)
            all_probs.extend(probs[:, 1].cpu().numpy())

    return all_true_labels, all_predictions, all_probs

# --- 1. Get All Predictions & Probabilities ---
y_true, y_pred, y_probs = get_predictions_and_probs(model_tl, test_loader_tl, device)

# --- 2. Generate Classification Report (Precision, Recall, F1) ---
class_names = ["REAL", "FAKE"] # Class 0 is REAL, Class 1 is FAKE
print("\n" + "="*30)
print("  FINAL CLASSIFICATION REPORT")
print("="*30)
print(classification_report(y_true, y_pred, target_names=class_names))


# --- 3. Generate and Plot Confusion Matrix ---
print("\n" + "="*30)
print("    FINAL CONFUSION MATRIX")
print("="*30)
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Final Model Confusion Matrix')
plt.ylabel('Actual (True) Label')
plt.xlabel('Predicted Label')
plt.show()

# --- 4. Generate and Plot ROC Curve ---
print("\n" + "="*30)
print("     FINAL ROC CURVE & AUC")
print("="*30)

# Calculate ROC curve
fpr, tpr, _ = roc_curve(y_true, y_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

print(f"Final AUC Score: {roc_auc:.4f}")

In [ ]:
#Cell:17#
# --- Save FINAL Best Model ---

final_model_path = os.path.join(capstone_path, "FINAL_finetuned_model_new.pth")
torch.save(model_tl.state_dict(), final_model_path)

print(f"--- SUCCESS! ---")
print(f"The model is now saved to:")
print(final_model_path)

In [ ]:
#Cell:18#
# --- Build model_tl and load that path ---
import torchvision.models as models
import torch.nn as nn
import torch
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1. Rebuild ResNet50 with a 2-class head
model_tl = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
num_ftrs = model_tl.fc.in_features
model_tl.fc = nn.Linear(num_ftrs, 2)

# 2. Use the exact path for best model
best_model_path = "/content/drive/MyDrive/Capstone/AUC 0.9704 - Run 2 - FINAL_finetuned_model_new.pth"

print("Loading checkpoint from:", best_model_path)
state_dict = torch.load(best_model_path, map_location=device)
model_tl.load_state_dict(state_dict)

model_tl = model_tl.to(device)
model_tl.eval()

print("Model loaded and set to eval mode.")


In [ ]:
#Cell:19#
#Rebuild test_loader_tl#
from torch.utils.data import DataLoader, random_split

print("--- Rebuilding FFT dataset and test loader for latency ---")
fft_full_dataset = FFTImageDataset(root_dir=new_dataset_path,
                                   transform=transfer_transform)

total_size = len(fft_full_dataset)
train_size = int(total_size * 0.5)
val_size = int(total_size * 0.25)
test_size = total_size - train_size - val_size

train_dataset_tl, val_dataset_tl, test_dataset_tl = random_split(
    fft_full_dataset, [train_size, val_size, test_size]
)

BATCH_SIZE = 16
test_loader_tl = DataLoader(test_dataset_tl, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total FFT images: {total_size}")
print(f"Test set size: {len(test_dataset_tl)}")


In [ ]:
#Cell 20#
#Latency measurement#
import time
import numpy as np

model = model_tl.to(device)
model.eval()

# Grab one batch from the test loader
inputs, labels = next(iter(test_loader_tl))
inputs = inputs.to(device)

print("Batch size:", inputs.shape[0])

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(inputs)

# Timed runs
times = []
iters = 50

with torch.no_grad():
    for _ in range(iters):
        start = time.perf_counter()
        _ = model(inputs)
        if device.type == "cuda":
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append(end - start)

times = np.array(times)
batch_ms = times * 1000.0
per_image_ms = batch_ms / inputs.shape[0]

print("Median batch time (ms):", np.median(batch_ms))
print("95th percentile batch time (ms):", np.percentile(batch_ms, 95))
print("Median per-image time (ms):", np.median(per_image_ms))
print("95th percentile per-image time (ms):", np.percentile(per_image_ms, 95))
